In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import pandas as pd


In [ ]:
# not set up yet
import wandb

# super_secret_API_key = "d1678741bcd8eb1d90ca26aa8f2bf079256ac391"

# wandb.login(key=super_secret_API_key)

# WANDB_PROJECT = "ZNEUS_project1"
# WANDB_ENTITY = None  # getpass.getuser()

In [ ]:
print(torch.__version__)
print(torch.version.cuda)
print(torch.randn(1).cuda())

print("CUDA available:", torch.cuda.is_available())


print("Current device index:", torch.cuda.current_device())
print("Device name:", torch.cuda.get_device_name(0))


2.9.1+cu130
13.0
tensor([0.7436], device='cuda:0')
CUDA available: True
Current device index: 0
Device name: NVIDIA GeForce RTX 3060 Laptop GPU


In [13]:
class Config:
    CSV_FILE = "images/sports.csv"
    ROOT_DIR = "images"
    BATCH_SIZE = 32
    NUM_EPOCHS = 10
    LEARNING_RATE = 0.001
    WEIGHT_DECAY = 1e-4

    # Data loading
    NUM_WORKERS = 2
    PIN_MEMORY = True

    # Device
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    PATIENCE = 10
    LR_SCHEDULER = "cosine"  # Options: "step", "cosine", "plateau"

    def __repr__(self):
        return "\n".join(
            [f"{k}: {v}" for k, v in self.__dict__.items() if not k.startswith("_")]
        )

In [14]:
class CustomCNN(nn.Module):
    def __init__(self, num_classes):
        super(CustomCNN, self).__init__()

        # Convolutional Block 1
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2, 2)  # 224 -> 112
        self.dropout1 = nn.Dropout2d(0.25)

        # Convolutional Block 2
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2, 2)  # 112 -> 56
        self.dropout2 = nn.Dropout2d(0.25)

        # Convolutional Block 3
        self.conv5 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(128)
        self.conv6 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.bn6 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(2, 2)  # 56 -> 28
        self.dropout3 = nn.Dropout2d(0.3)

        # Convolutional Block 4
        self.conv7 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn7 = nn.BatchNorm2d(256)
        self.conv8 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.bn8 = nn.BatchNorm2d(256)
        self.pool4 = nn.MaxPool2d(2, 2)  # 28 -> 14
        self.dropout4 = nn.Dropout2d(0.3)

        # Global Average Pooling
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))

        # Fully Connected Layers
        self.fc1 = nn.Linear(256, 512)
        self.bn_fc1 = nn.BatchNorm1d(512)
        self.dropout5 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        # Block 1
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool1(x)
        x = self.dropout1(x)

        # Block 2
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = self.pool2(x)
        x = self.dropout2(x)

        # Block 3
        x = F.relu(self.bn5(self.conv5(x)))
        x = F.relu(self.bn6(self.conv6(x)))
        x = self.pool3(x)
        x = self.dropout3(x)

        # Block 4
        x = F.relu(self.bn7(self.conv7(x)))
        x = F.relu(self.bn8(self.conv8(x)))
        x = self.pool4(x)
        x = self.dropout4(x)

        # Global Average Pooling
        x = self.global_avg_pool(x)
        x = x.view(x.size(0), -1)

        # Fully Connected Layers
        x = F.relu(self.bn_fc1(self.fc1(x)))
        x = self.dropout5(x)
        x = self.fc2(x)

        return x


def create_model(num_classes):
    model = CustomCNN(num_classes)
    return model

In [15]:
class SportsDataset(Dataset):
    def __init__(self, csv_file, root_dir, dataset_type, transform=None):
        df = pd.read_csv(csv_file)
        self.df = df[df["data set"] == dataset_type]
        self.root_dir = root_dir
        self.transform = transform

        self.classes = sorted(self.df["labels"].unique())
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.root_dir, row["filepaths"])
        image = Image.open(img_path).convert("RGB")

        label = self.class_to_idx[row["labels"]]

        if self.transform:
            image = self.transform(image)

        return image, label


In [16]:
train_transform = transforms.Compose(
    [
        transforms.Resize((256, 256)),
        transforms.RandomCrop((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

test_transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

trainset = SportsDataset("images/sports.csv", "images", "train", train_transform)
validset = SportsDataset("images/sports.csv", "images", "valid", test_transform)
testset = SportsDataset("images/sports.csv", "images", "test", test_transform)

trainloader = DataLoader(trainset, batch_size=32, shuffle=True)
validloader = DataLoader(validset, batch_size=32, shuffle=False)
testloader = DataLoader(testset, batch_size=32, shuffle=False)


In [17]:
config = Config()
model = CustomCNN(num_classes=len(trainset.classes))
model = model.to(config.DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)


In [18]:
def train(
    model, trainloader, validloader, criterion, optimizer, scheduler, device, epochs=20
):
    best_loss = float("inf")

    for epoch in range(epochs):
        print("begun epoch", epoch)
        model.train()
        train_loss = 0

        for images, labels in trainloader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)

        train_loss /= len(trainloader.dataset)

        # ---- validation ----
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for images, labels in validloader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)

        val_loss /= len(validloader.dataset)

        scheduler.step()

        print(
            f"Epoch {epoch + 1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}"
        )

        # save best model
        if val_loss < best_loss:
            best_loss = val_loss
            torch.save(model.state_dict(), "./model")
            print("  → Saved best model")

    print("Training finished!")


In [19]:
def test(model, testloader, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

    print(f"Test Accuracy: {correct / total:.4f}")


In [20]:
config.DEVICE

device(type='cuda')

In [21]:
train(
    model,
    trainloader,
    validloader,
    criterion,
    optimizer,
    scheduler,
    config.DEVICE,
    config.NUM_EPOCHS,
)


begun epoch 0
Epoch 1 | Train Loss: 4.3026 | Val Loss: 3.7974
  → Saved best model
begun epoch 1
Epoch 2 | Train Loss: 3.9051 | Val Loss: 3.5869
  → Saved best model
begun epoch 2
Epoch 3 | Train Loss: 3.7231 | Val Loss: 3.3486
  → Saved best model
begun epoch 3
Epoch 4 | Train Loss: 3.6223 | Val Loss: 3.3101
  → Saved best model
begun epoch 4
Epoch 5 | Train Loss: 3.4965 | Val Loss: 3.1022
  → Saved best model
begun epoch 5
Epoch 6 | Train Loss: 3.4008 | Val Loss: 3.0559
  → Saved best model
begun epoch 6
Epoch 7 | Train Loss: 3.3126 | Val Loss: 2.8791
  → Saved best model
begun epoch 7
Epoch 8 | Train Loss: 3.2398 | Val Loss: 2.8486
  → Saved best model
begun epoch 8
Epoch 9 | Train Loss: 3.1490 | Val Loss: 2.7967
  → Saved best model
begun epoch 9
Epoch 10 | Train Loss: 3.0779 | Val Loss: 2.6468
  → Saved best model
Training finished!


FileNotFoundError: [Errno 2] No such file or directory: 'best_model.pth'

In [23]:
model.load_state_dict(torch.load("./model"))
test(model, testloader, config.DEVICE)

Test Accuracy: 0.2940
